In [18]:
import numpy as np
import scipy.stats as stats


# Model Parameters
S0 = 100        # Start price
K = 100         # Strike
L = 120         # Barrier level
T = 1.0         # Duration (in years)
r = 0.05        # risk-free interst rate
sigma = 0.2     # Volatility
d = 0.03        # Dividend yield (continuous)

# Monte Carlo Parameter
n_sim = 100000      # Number of simulations
n_steps = 252       # Number of time discretisations (1 day)
dt = T / n_steps    # Timestep
D = 10              # Parisian time-window D

#GBM in python code with antithetic variates
#matrix with standard-normally distributed random numbers 
Z = np.random.normal(size=(n_sim, n_steps))
Z_anti = -Z  # antithetic counterpart

S = np.zeros((n_sim, n_steps)) #structure for all the simulated price history
S_anti = np.zeros((n_sim, n_steps))

S[:, 0] = S0 #every first column of each simulation starts with value S_0
S_anti[:, 0] = S0


for t in range(1, n_steps):
    S[:, t] = S[:, t - 1] * np.exp((r - d - 0.5 * sigma**2) 
    * dt + sigma * np.sqrt(dt) * Z[:, t]) #GBM equation with dividend 
    S_anti[:, t] = S_anti[:, t - 1] * np.exp((r - d - 0.5 * sigma**2) 
    * dt + sigma * np.sqrt(dt) * Z_anti[:, t]) #GBM equation with dividend                                                                                                  #Dividdens from Glassman, 2003, P.97


def parisian_knock(S, L, D):
  
    n_sim, n_steps = S.shape
    knock = np.zeros(n_sim, dtype=bool)

    for i in range(n_sim):
        consecutive = 0   #counts the length of the current "Excursion" above B.
        for t in range(n_steps):   #The current price S_{i,t} is above the barrier --> Excursion continues → count days.
            if S[i, t] >= L:
                consecutive += 1
                if consecutive >= D:
                    knock[i] = True
                    break
            else:
                consecutive = 0  # Reset if price falls below barrier

    return knock


knock = parisian_knock(S, L, D)
knock_anti = parisian_knock(S_anti, L, D)


#python code for payoff and discounted price
payoff = np.where(knock, 0.0, np.maximum(S[:, -1] - K, 0.0))
payoff_anti = np.where(knock_anti, 0.0, np.maximum(S_anti[:, -1] - K, 0.0))

# Average both payoffs for variance reduction
payoff_avg = 0.5 * (payoff + payoff_anti)

# Discounted Price (Antithetic Estimator)
discounted_price = np.exp(-r * T) * np.mean(payoff_avg)

#print
# Output
print (f"Discountet Price: {discounted_price:.4f}")


Discountet Price: 2.1368
